# Stage 3 CLIP / Hybrid Coarse Benchmark Clean (Kaggle)

Coarse-only benchmark for the hybrid direction: a discriminative visual classifier predicts `coarse_class`, while Qwen remains the structured reporter in the broader system.

This notebook does not replace `vlm_labels_v1` generation. It evaluates whether CLIP-style visual embeddings can improve the hard `insulator_ok` vs `defect_flashover` boundary.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path
from collections import Counter

import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"

RUN_NAME = "stage3_clip_hybrid_coarse_clean"
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]
TEXT_PROMPTS = {
    "insulator_ok": "a clear photo crop of an intact power line insulator with regular shape and no visible damage",
    "defect_flashover": "a photo crop of a power line insulator with flashover burn marks dark surface traces or localized surface damage",
    "defect_broken": "a photo crop of a broken power line insulator with missing fragment damaged edge or structural break",
}
print("RUN_NAME:", RUN_NAME)


In [ ]:
def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path: Path):
    rows=[]
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def resolve_crop(row, split_root: Path) -> Path:
    p = Path(row["crop_path"])
    if p.is_absolute() and p.exists():
        return p
    cand = split_root / p
    if cand.exists():
        return cand
    cand = split_root.parent / p
    if cand.exists():
        return cand
    raise FileNotFoundError(f"crop not found: {row['crop_path']}")


In [ ]:
gpu = sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    print("GPU not available; continuing on CPU will be slower.")
else:
    print("GPU available")


In [ ]:
DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root / TRAIN_JSONL_REL).exists() and (root / VAL_JSONL_REL).exists():
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("Could not find stage3_regrouped_v2 train/val JSONL in Kaggle inputs")

train_jsonl = DATA_ROOT / TRAIN_JSONL_REL
val_jsonl = DATA_ROOT / VAL_JSONL_REL
train_root = train_jsonl.parent
val_root = val_jsonl.parent
print("DATA_ROOT:", DATA_ROOT)
print("train_jsonl:", train_jsonl)
print("val_jsonl:", val_jsonl)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])
sh(["python", "-m", "pip", "install", "-q", "-U", "transformers", "pillow", "pandas", "scikit-learn", "tabulate"], cwd=REPO_DIR)
print("Repo ready:", REPO_DIR)


In [ ]:
import numpy as np
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import normalize
from transformers import CLIPModel, CLIPProcessor

train_rows = [r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
val_rows = [r for r in load_jsonl(val_jsonl) if r.get("coarse_class") in LABELS]
print("train rows:", len(train_rows), Counter(r["coarse_class"] for r in train_rows))
print("val rows:", len(val_rows), Counter(r["coarse_class"] for r in val_rows))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
model.eval()
print("device:", device)


In [ ]:
def embed_images(rows, split_root: Path, batch_size: int = 32):
    feats=[]
    labels=[]
    ids=[]
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start+batch_size]
            images=[]
            for row in batch:
                img = Image.open(resolve_crop(row, split_root)).convert("RGB")
                images.append(img)
            inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
            emb = model.get_image_features(**inputs)
            emb = emb.detach().cpu().numpy()
            feats.append(emb)
            labels.extend([r["coarse_class"] for r in batch])
            ids.extend([r["record_id"] for r in batch])
    feats = normalize(np.vstack(feats))
    return feats, np.array(labels), ids

def embed_texts():
    texts=[TEXT_PROMPTS[label] for label in LABELS]
    with torch.no_grad():
        inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
        emb = model.get_text_features(**inputs).detach().cpu().numpy()
    return normalize(emb)

X_train, y_train, train_ids = embed_images(train_rows, train_root)
X_val, y_val, val_ids = embed_images(val_rows, val_root)
T = embed_texts()
print("embeddings:", X_train.shape, X_val.shape, T.shape)


In [ ]:
# Zero-shot CLIP ranking
sim = X_val @ T.T
zs_pred = np.array([LABELS[i] for i in sim.argmax(axis=1)])

# Linear probe on train_balanced embeddings
clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, multi_class="auto")
clf.fit(X_train, y_train)
lp_pred = clf.predict(X_val)

# Hybrid safety variant: use linear probe unless confidence is low, then abstain as unknown.
proba = clf.predict_proba(X_val)
maxp = proba.max(axis=1)
hybrid_pred = np.array([p if conf >= 0.45 else "unknown" for p, conf in zip(lp_pred, maxp)])

def macro_f1_with_unknown(y_true, y_pred):
    labels = ["insulator_ok", "defect_flashover", "defect_broken", "unknown", "other"]
    return f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

def summarize(name, pred):
    return {
        "method": name,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1_3class": f1_score(y_val, pred, labels=LABELS, average="macro", zero_division=0),
        "macro_f1_with_unknown": macro_f1_with_unknown(y_val, pred),
        "ok_recall": ((pred == "insulator_ok") & (y_val == "insulator_ok")).sum() / max((y_val == "insulator_ok").sum(), 1),
        "flashover_recall": ((pred == "defect_flashover") & (y_val == "defect_flashover")).sum() / max((y_val == "defect_flashover").sum(), 1),
        "broken_recall": ((pred == "defect_broken") & (y_val == "defect_broken")).sum() / max((y_val == "defect_broken").sum(), 1),
    }

summary = pd.DataFrame([
    summarize("clip_zero_shot", zs_pred),
    summarize("clip_linear_probe", lp_pred),
    summarize("clip_linear_probe_unknown_threshold_045", hybrid_pred),
])
display(summary)

for name, pred in [("clip_zero_shot", zs_pred), ("clip_linear_probe", lp_pred), ("clip_linear_probe_unknown_threshold_045", hybrid_pred)]:
    print("\n", name)
    print(classification_report(y_val, pred, labels=["insulator_ok", "defect_flashover", "defect_broken", "unknown"], zero_division=0))


In [ ]:
out_dir = REPO_DIR / "outputs" / RUN_NAME
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / "coarse_benchmark_summary.csv", index=False)

pred_rows=[]
for rid, gt, zs, lp, hp, conf in zip(val_ids, y_val, zs_pred, lp_pred, hybrid_pred, maxp):
    pred_rows.append({
        "record_id": rid,
        "gt": gt,
        "clip_zero_shot": zs,
        "clip_linear_probe": lp,
        "clip_linear_probe_confidence": float(conf),
        "clip_linear_probe_unknown_threshold_045": hp,
    })
pd.DataFrame(pred_rows).to_csv(out_dir / "coarse_predictions.csv", index=False)

lines = [
    "# Stage 3 CLIP / Hybrid Coarse Benchmark Clean",
    "",
    f"CLIP model: `{CLIP_MODEL_ID}`",
    "",
    summary.to_markdown(index=False),
    "",
    "TL-CLIP status: not executed because no public runnable weights were attached to this Kaggle run.",
]
(out_dir / "summary.md").write_text("\n".join(lines), encoding="utf-8")

archive_path = Path("/kaggle/working/stage3_deliverables_clip_hybrid_coarse_clean.tar.gz")
if archive_path.exists():
    archive_path.unlink()
sh(["tar", "-czf", str(archive_path), "-C", str(out_dir.parent), RUN_NAME], check=True)
print("Archive:", archive_path)
